# The cross-entropy method

The cross-entropy (CE) method treats optimization as repeated importance sampling:

1. sample candidate solutions from a parametric distribution;
2. score them and keep the best fraction (the *elite*);
3. refit the distribution to the elite;
4. repeat until the distribution concentrates.

I use a logit-normal distribution for portfolio weights (the softmax keeps weights positive and summing to one) and an edge-probability matrix for the travelling salesman problem.


In [1]:
import numpy as np


def sharpe_ratio(weights, expected_returns, covariance, risk_free_rate=0.0):
    """Annualized Sharpe ratio for a vector of portfolio weights."""
    weights = np.asarray(weights, dtype=float)
    expected = weights @ expected_returns
    variance = weights @ covariance @ weights
    return (expected - risk_free_rate) / np.sqrt(variance)


def sharpe_ratios(samples, expected_returns, covariance):
    """Vectorized Sharpe ratio for a batch of weight vectors."""
    expected = samples @ expected_returns
    variance = np.einsum("ni,ij,nj->n", samples, covariance, samples)
    return expected / np.sqrt(variance)


def cross_entropy_portfolio(expected_returns, covariance, num_samples=4000,
                            num_iterations=100, elite_fraction=0.1,
                            risk_free_rate=0.0, seed=2024):
    """Maximize the Sharpe ratio over the probability simplex."""
    rng = np.random.default_rng(seed)
    n = len(expected_returns)
    mean = np.zeros(n)
    std = np.ones(n)
    num_elite = max(1, int(num_samples * elite_fraction))
    best_weights, best_score = None, -np.inf

    for _ in range(num_iterations):
        z = rng.normal(mean, std, size=(num_samples, n))
        z -= z.max(axis=1, keepdims=True)
        weights = np.exp(z)
        weights /= weights.sum(axis=1, keepdims=True)

        scores = sharpe_ratios(weights, expected_returns, covariance)
        elite = np.argsort(scores)[-num_elite:]
        mean = z[elite].mean(axis=0)
        std = z[elite].std(axis=0) + 1e-4

        if scores[elite[-1]] > best_score:
            best_score = scores[elite[-1]]
            best_weights = weights[elite[-1]].copy()

    return best_weights, best_score


expected_returns = np.array([0.12, 0.10, 0.15])
covariance = np.array([[0.10, 0.02, 0.04],
                       [0.02, 0.08, 0.01],
                       [0.04, 0.01, 0.12]])

weights, score = cross_entropy_portfolio(expected_returns, covariance)
print(f"weights: {np.round(weights, 4)}")
print(f"sum of weights: {weights.sum():.6f}")
print(f"Sharpe ratio: {score:.4f}")


weights: [0.2426 0.3812 0.3762]
sum of weights: 1.000000
Sharpe ratio: 0.5623


In [2]:
mu = np.array([0.2083, 0.1874, 0.2119, 0.1095, 0.0239, 0.001])

Sigma = np.array([
    [106.991461, 23.892217, -27.843578, -59.063559, 6.484768, 0.007298],
    [23.892217, 98.632733, -117.56474, 226.16329, 16.106375, -0.006264],
    [-27.843578, -117.56474, 206.012325, -40.45979, -6.732048, -0.006654],
    [-59.063559, 226.16329, -40.45979, 1610.96212, -6.718355, -0.000654],
    [6.484768, 16.106375, -6.732048, -6.718355, 1.234519, -0.0009],
    [0.007298, -0.006264, -0.006654, -0.000654, -0.0009, 0.000013]])

weights, score = cross_entropy_portfolio(mu, Sigma, seed=2024)
print(f"weights: {np.round(weights, 4)}")
print(f"Sharpe ratio: {score:.4f}")


weights: [1.820e-02 9.960e-02 7.370e-02 2.000e-04 1.600e-03 8.066e-01]
Sharpe ratio: 0.0595


In [3]:
import itertools


def route_distance(route, distance):
    route = np.asarray(route)
    return distance[route, np.roll(route, -1)].sum()


def cross_entropy_tsp(distance, num_samples=1000, num_iterations=60,
                      elite_fraction=0.2, learning_rate=0.5, seed=2024):
    """CE for the TSP: sample tours from an edge-probability matrix."""
    rng = np.random.default_rng(seed)
    n = distance.shape[0]
    probabilities = np.ones((n, n)) / (n - 1)
    np.fill_diagonal(probabilities, 0.0)
    best_distance, best_route = np.inf, None
    num_elite = max(1, int(num_samples * elite_fraction))

    for _ in range(num_iterations):
        routes = []
        for _ in range(num_samples):
            route = [0]
            unvisited = set(range(1, n))
            while unvisited:
                candidates = np.array(sorted(unvisited))
                p = probabilities[route[-1], candidates]
                p = p / p.sum()
                next_city = int(rng.choice(candidates, p=p))
                route.append(next_city)
                unvisited.remove(next_city)
            routes.append(route)

        distances = np.array([route_distance(r, distance) for r in routes])
        elite = np.argsort(distances)[:num_elite]

        frequency = np.zeros((n, n))
        for index in elite:
            route = routes[index]
            for a, b in zip(route, np.roll(route, -1)):
                frequency[a, b] += 1
        frequency /= num_elite

        probabilities = (1 - learning_rate) * probabilities + learning_rate * frequency
        np.fill_diagonal(probabilities, 0.0)
        probabilities /= probabilities.sum(axis=1, keepdims=True)

        if distances[elite[0]] < best_distance:
            best_distance = distances[elite[0]]
            best_route = routes[elite[0]]

    return best_route, best_distance


distance = np.array([[0, 10, 15, 20],
                     [10, 0, 35, 25],
                     [15, 35, 0, 30],
                     [20, 25, 30, 0]], dtype=float)

best_route, best_distance = cross_entropy_tsp(distance)
print(f"best route: {best_route}")
print(f"route length: {best_distance}")

brute_force = min(route_distance(p, distance)
                  for p in itertools.permutations(range(distance.shape[0]))
                  if p[0] == 0)
print(f"brute-force optimum: {brute_force}")


best route: [0, 1, 3, 2]
route length: 80.0
brute-force optimum: 80.0


## What to expect

CE is a stochastic search: a single run may return a near-optimal tour or a portfolio weight close to the boundary of the simplex. Fixing the seed makes the saved output reproducible; the brute-force check shows that the 4-city tour found here is optimal.
